# DWS 层 — 日粒度聚合

`fact_trips` → 按日 × 区域 × 来源 聚合，产出 `dws/trips_daily`。

两个核心优化：
1. **加盐两阶段聚合**：热点区域（如 161/237）数据倾斜，加随机盐先局部聚合，再合并
2. **滑动窗口处理跨日迟到数据**：重算时往前多取一天，覆盖迟到记录

## 1. SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, year, month, dayofmonth

spark = (
    SparkSession.builder
    .appName("TLC-DWS-Build")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE     = os.path.expanduser("~/tlc-spark-pipeline")
DWD_FACT = f"{BASE}/data/warehouse/dwd/fact_trips"
DWS_OUT  = f"{BASE}/data/warehouse/dws/trips_daily"

TARGET_MONTH = os.environ.get("TARGET_MONTH", "2024-03")
yr, mo = int(TARGET_MONTH.split("-")[0]), int(TARGET_MONTH.split("-")[1])
print(f"TARGET_MONTH={TARGET_MONTH}")

print(f"Spark {spark.version} ready")

## 2. 读取 fact_trips + 确认热点区域倾斜

In [2]:
fact = spark.read.parquet(DWD_FACT).filter(
    (col("pickup_year") == yr) & (col("pickup_month") == mo))
print(f"fact_trips: {fact.count():,} rows")

print("\n=== 热点区域（Top 10 上车量）===")
fact.groupBy("PULocationID") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy(F.desc("trip_count")) \
    .limit(10) \
    .show()

## 3. 加盐两阶段聚合（解决热点区域数据倾斜）

热点区域（如 161 号区域拥有 40 万+ 条记录）会导致 groupBy 时某个 partition 数据量远超其他，形成长尾。

**解决方案：**
1. 第一阶段：给 PULocationID 加随机盐（0~7），先做局部聚合，将一个大 key 拆成 8 个小 key
2. 第二阶段：去掉盐，再做一次合并聚合，得到真实的每日区域汇总

In [3]:
SALT_BUCKETS = 8

# 增加日期列
fact_with_date = fact.withColumn("pickup_date", to_date("pickup_datetime"))

# 第一阶段：加盐局部聚合
phase1 = (
    fact_with_date
    .withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))
    .groupBy("pickup_date", "PULocationID", "source_type", "salt")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("gmv"),
        F.sum("fare_amount").alias("fare_sum"),
        F.sum("tip_amount").alias("tip_sum"),
        F.sum("surcharge_total").alias("surcharge_sum"),
        F.sum("trip_distance").alias("distance_sum"),
        F.sum("trip_duration_min").alias("duration_sum")
    )
)

# 第二阶段：去盐合并
dws = (
    phase1
    .groupBy("pickup_date", "PULocationID", "source_type")
    .agg(
        F.sum("trip_count").alias("trip_count"),
        F.round(F.sum("gmv"), 2).alias("gmv"),
        F.round(F.sum("fare_sum"), 2).alias("fare_sum"),
        F.round(F.sum("tip_sum"), 2).alias("tip_sum"),
        F.round(F.sum("surcharge_sum"), 2).alias("surcharge_sum"),
        F.round(F.sum("distance_sum"), 2).alias("distance_sum"),
        F.round(F.sum("duration_sum"), 2).alias("duration_sum")
    )
    .withColumn("pickup_year",  year("pickup_date"))
    .withColumn("pickup_month", month("pickup_date"))
    .withColumn("pickup_day",   dayofmonth("pickup_date"))
)

dws_count = dws.count()
print(f"DWS rows: {dws_count:,}")
dws.show(5)

## 4. 滑动窗口处理跨日迟到数据

按日增量跑任务时，某些行程的数据可能在次日才到达（如深夜行程跨零点上报）。

**处理方式：** 重算 T 日时，实际查询范围是 `[T-1 23:00, T+1 01:00]`，覆盖跨日边界，然后只输出 pickup_date = T 的记录，通过动态分区覆盖保证幂等。

In [4]:
# 演示：以 2024-01-15 为目标日期，展示滑动窗口逻辑
TARGET_DATE = "2024-01-15"

dws_single_day = (
    fact_with_date
    # 滑动窗口：取前后各 1 小时，覆盖跨日迟到
    .filter(col("pickup_datetime") >= F.expr(f"timestamp('{TARGET_DATE}') - interval 1 hours"))
    .filter(col("pickup_datetime") <  F.expr(f"timestamp('{TARGET_DATE}') + interval 25 hours"))
    # 只保留目标日期的记录输出
    .filter(col("pickup_date") == TARGET_DATE)
    .withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))
    .groupBy("pickup_date", "PULocationID", "source_type", "salt")
    .agg(F.count("*").alias("trip_count"), F.sum("total_amount").alias("gmv"),
         F.sum("fare_amount").alias("fare_sum"), F.sum("tip_amount").alias("tip_sum"),
         F.sum("surcharge_total").alias("surcharge_sum"),
         F.sum("trip_distance").alias("distance_sum"), F.sum("trip_duration_min").alias("duration_sum"))
    .groupBy("pickup_date", "PULocationID", "source_type")
    .agg(F.sum("trip_count").alias("trip_count"), F.round(F.sum("gmv"), 2).alias("gmv"),
         F.round(F.sum("fare_sum"), 2).alias("fare_sum"), F.round(F.sum("tip_sum"), 2).alias("tip_sum"),
         F.round(F.sum("surcharge_sum"), 2).alias("surcharge_sum"),
         F.round(F.sum("distance_sum"), 2).alias("distance_sum"),
         F.round(F.sum("duration_sum"), 2).alias("duration_sum"))
    .withColumn("pickup_year",  year("pickup_date"))
    .withColumn("pickup_month", month("pickup_date"))
    .withColumn("pickup_day",   dayofmonth("pickup_date"))
)

print(f"{TARGET_DATE} zones: {dws_single_day.count()}")
dws_single_day.orderBy(F.desc("trip_count")).show(5)

## 5. 写出 DWS（动态分区覆盖）

In [5]:
%%time
(
    dws
    .write
    .partitionBy("pickup_year", "pickup_month", "pickup_day")
    .mode("overwrite")
    .parquet(DWS_OUT)
)
print(f"DWS written to: {DWS_OUT}")

# 验证
dws_check = spark.read.parquet(DWS_OUT)
print(f"DWS total rows: {dws_check.count():,}")
dws_check.groupBy("pickup_year", "pickup_month", "source_type") \
    .agg(F.sum("trip_count").alias("trips"), F.round(F.sum("gmv"), 0).alias("gmv")) \
    .orderBy("pickup_year", "pickup_month", "source_type").show()

spark.stop()
print("DWS build complete.")